# Video Swin en Colab: carga, inferencia y ajuste fino

Usa el paquete `vswin` (carpeta `Video-Swin-Transformer-Colab`). No necesita mmcv ni mmaction.

Antes de empezar: **Entorno de ejecución → Cambiar tipo → GPU**, y sube a Drive:
- la carpeta `Video-Swin-Transformer-Colab/`
- la salida del preprocesamiento `salida/<conjunto>/` (con `miniclips/` y `meta/`)

In [ ]:
# Rutas en Drive (ajústalas)
PKG_DIR   = '/content/drive/MyDrive/Video-Swin-Transformer-Colab'
DATA_DRIVE = '/content/drive/MyDrive/salida/pornography2k'   # salida/<conjunto>
DATA_LOCAL = '/content/data/pornography2k'                   # copia en el disco local de Colab (lectura más rápida)
RUN_DIR   = '/content/drive/MyDrive/vswin_runs/p2k_swin_b_fold1'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q "$PKG_DIR"
!ffmpeg -hide_banner -version | head -n 1

In [ ]:
import torch, vswin
print(vswin.describe_environment())
print(vswin.list_variants())

## 1. Cargar el modelo preentrenado y comprobar la entrada

Entrada: tensor `float32` **(B, 3, 32, 224, 224)** en RGB, normalizado con la media y desviación de ImageNet.
El checkpoint se descarga una sola vez a `MyDrive/vswin_cache/`.

In [ ]:
device = vswin.default_device()

# Modelo completo de Kinetics-400 (400 clases): sirve para verificar la carga
k400 = vswin.build_model('swin_b_k400_22k', device=device).eval()
print(vswin.count_parameters(k400))

x = vswin.dummy_input(batch=1, device=device)        # (1, 3, 32, 224, 224)
with torch.no_grad(), torch.autocast('cuda', dtype=vswin.amp_dtype()):
    feat = k400.forward_features(x, pooled=False)
    logits = k400(x)
print('backbone:', tuple(feat.shape), ' logits:', tuple(logits.shape))   # (1, 1024, 16, 7, 7)  (1, 400)
print(f'GPU memory peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB')

del k400, x, feat, logits
torch.cuda.empty_cache()

## 2. Datos: miniclips del preprocesamiento

Copiar a `/content` evita leer cada `.mp4` desde Drive en cada época.
Las etiquetas pueden venir de un dict `{video_id: clase}`, de un CSV (`video_id,label`) o de la ruta original (`meta['ruta']`).
En Pornography-2k los archivos se llaman `vPorn…` / `vNonPorn…`.

In [ ]:
!mkdir -p "$DATA_LOCAL" && rsync -a --info=progress2 "$DATA_DRIVE/" "$DATA_LOCAL/"

CLASS_NAMES = ['no_explicito', 'explicito']
records = vswin.index_dataset(DATA_LOCAL, labels=vswin.label_from_path({'NonPorn': 0, 'Porn': 1}))
# records = vswin.index_dataset(DATA_LOCAL, labels='/content/drive/MyDrive/etiquetas.csv', class_names=CLASS_NAMES)

# Partición por video. Para los folds oficiales usa vswin.select_records(records, 'fold1_train.txt')
train_rec, val_rec = vswin.split_records(records, val_fraction=0.2, seed=0)
print('train:', vswin.describe_records(train_rec, CLASS_NAMES))
print('val:  ', vswin.describe_records(val_rec, CLASS_NAMES))

train_loader, val_loader = vswin.make_loaders(
    train_rec, val_rec,
    batch_size=2,          # miniclips por paso (T4: 2 con gradient checkpointing)
    num_workers=2,
    flip_prob=0.5,         # volteo horizontal, como el pipeline oficial
    balanced_sampling=True # cada video y cada clase pesan lo mismo
)
batch = next(iter(train_loader))
print(batch['clip'].shape, batch['clip'].dtype, batch['label'])   # (2, 32, 224, 224, 3) uint8

## 3. Ajuste fino a 2 clases

Se carga solo el backbone preentrenado; la cabeza (1024 → 2) empieza desde cero.
Si Colab se desconecta, vuelve a ejecutar esta sección: `FineTuner` continúa desde `RUN_DIR/last.pth`.

In [ ]:
vswin.set_seed(0)
model = vswin.build_model('swin_b_k400_22k', num_classes=2, use_checkpoint=True, device=device)

cfg = vswin.FineTuneConfig(
    output_dir=RUN_DIR,
    epochs=30,
    lr=3e-4,                  # cabeza; backbone = lr × 0.1
    backbone_lr_mult=0.1,
    weight_decay=0.05,
    warmup_epochs=2.5,
    accumulation_steps=8,     # lote efectivo = 2 × 8 = 16 miniclips
    freeze_backbone_epochs=0,
    monitor='video_acc',
    class_names=CLASS_NAMES,
)
tuner = vswin.FineTuner(model, train_loader, val_loader, cfg, device=device)
history = tuner.fit()

## 4. Evaluar el mejor modelo

In [ ]:
tuner.load_best()
metrics, rows = vswin.evaluate(model, val_loader, device)
for k, v in metrics.items():
    print(f'{k:22s} {v:.4f}')
cm = vswin.confusion_matrix([r['label'] for r in rows], [r['pred'] for r in rows], 2)
print('confusion matrix (rows = true):\n', cm)

## 5. Predecir un video y extraer características

In [ ]:
# Cargar el modelo afinado en una sesión nueva:
# model = vswin.build_model('swin_b_k400_22k', num_classes=2, pretrained=f'{RUN_DIR}/best.pth', device=device)

rec = val_rec[0]
clips = vswin.read_miniclips(rec.clip_paths)          # (K, 32, 224, 224, 3) uint8
pred = vswin.predict_clips(model, clips, batch_size=4)
print(rec.video_id, 'label', rec.label, '->', pred.top_k(2, CLASS_NAMES))

feats = vswin.extract_features(model, clips, reduce=None)  # (K, 1024)
print(feats.shape)